last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [53]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [55]:
import os


In [56]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [57]:
#! curl ipinfo.io

In [58]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [7]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [59]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_8651/128205925.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [60]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [61]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [62]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [63]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [64]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [65]:
from langchain_community.document_loaders import WebBaseLoader

In [66]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

In [67]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [68]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [69]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [70]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [71]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.
Madam Speaker, Madam Vice President, our First Lady and Seco

각 chunk의 길이를 확인해보겠습니다,

In [72]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [73]:
!pip install tiktoken

In [74]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [75]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]
[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [76]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [77]:
client = openai.OpenAI()

In [78]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large
text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [79]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [80]:
#!curl ipinfo.io

In [81]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [82]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [83]:
print(embeddings[1])

[0.006427764892578125, -0.020843505859375, -0.0216217041015625, 0.0235443115234375, -0.0428466796875, -0.004589080810546875, 0.039703369140625, 0.05267333984375, -0.0018796920776367188, -0.0237579345703125, 0.0138092041015625, 0.0159912109375, -0.044464111328125, -0.00011223554611206055, -0.007648468017578125, 0.02557373046875, 0.025421142578125, 0.03546142578125, -0.051788330078125, 0.01232147216796875, 0.02484130859375, -0.00047087669372558594, 0.0193939208984375, 0.07525634765625, -0.002044677734375, -0.009185791015625, 0.0168304443359375, -0.007129669189453125, 0.02655029296875, -0.04852294921875, 0.043182373046875, -0.042572021484375, 0.01163482666015625, -0.03289794921875, -0.0333251953125, -0.04620361328125, 0.0019054412841796875, 0.0225677490234375, -0.018096923828125, 0.03253173828125, 0.0293426513671875, 0.011749267578125, -0.01446533203125, 0.003063201904296875, 0.024322509765625, 0.04815673828125, -0.01354217529296875, 0.04937744140625, -0.0406494140625, 0.04400634765625, 0

In [84]:
len(embeddings[1])

1536

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [85]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [86]:
query = ["this is red fruit"]

In [87]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478929665954975
0.4898791411166917
0.4083791543048118
0.7478929665954975
0.4898791411166917
0.4083791543048118


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [88]:
!pip install chromadb

In [89]:
!pip install langchain-chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)


In [90]:
from langchain_chroma import Chroma

In [91]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [92]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [93]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [94]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [95]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [96]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [97]:
!pip install -U langchain langchain-classic

In [98]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [99]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [100]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [101]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_8651/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. The narrator perceives Demian as being like an animal, a spirit, or an image, and describes him as unimaginably different from others.Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face is described as having an almost feminine element, and it is suggested that he looks different from the rest of the people, almost like an animal, a spirit, or an image. The narrator finds it difficult to describe him precisely, only noting that he is unimaginably different from others.

마크다운 형식으로 출력해봅니다

In [102]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. The narrator perceives Demian as being like an animal, a spirit, or an image, and describes him as unimaginably different from others.

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face is described as having an almost feminine element, and it is suggested that he looks different from the rest of the people, almost like an animal, a spirit, or an image. The narrator finds it difficult to describe him precisely, only noting that he is unimaginably different from others.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [103]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you're referring to the character Demian from Hermann Hesse's novel "Demian: The Story of Emil Sinclair's Youth," there isn't an exact description provided in the book, allowing for some interpretation. However, certain elements can be gleaned from the text and its themes:

1. **Symbolic Appearance**: Demian often embodies duality and the concept of embracing both good and bad within oneself. This duality might be reflected in his appearance as being both ordinary and extraordinary or possessing a magnetic presence.

2. **Piercing Eyes**: One notable feature often highlighted in character analyses or artistic interpretations is Demian's penetrating eyes, which seem to convey deep understanding and wisdom.

3. **Youthful but Mature**: While he is youthful in age, Demian carries an aura of wisdom and maturity beyond his years, which might manifest in his demeanor, posture, or expression.

4. **Ambiguous and Intriguing Look**: As a character who often challenges societal norms and perceptions, his appearance might also be subtly unconventional, intriguing, or enigmatic.

Ultimately, Demian's appearance is left to the imagination of the reader, and various adaptations or illustrations may depict him differently based on the artist's or interpreter's perspective.

Demian is a character from Hermann Hesse's novel "Demian: The Story of Emil Sinclair's Youth." While the book doesn't provide a detailed physical description of Demian, his presence and personality leave a significant impact on the protagonist, Emil Sinclair, and readers alike. 

Demian is often portrayed as having an enigmatic and charismatic presence, with a somewhat otherworldly or intense aura. He is depicted as a figure who exudes confidence, wisdom, and the ability to see beyond conventional morality. His eyes are often noted for their penetrating and insightful gaze, which seems to understand the deepest thoughts and fears of those around him.

The lack of a detailed physical description allows readers to imagine Demian in a way that fits his mythical and symbolic role within the novel. His character is more defined by his philosophical ideas and influence on Sinclair's journey of self-discovery than by his physical appearance.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

결과와 함께 AI가 참고한 '원본 문서의 출처(Source Documents)'와 '검색된 문장(Context)'을 직접 눈으로 관찰할 수 있기 때문입니다. 근거가 명확하므로 AI가 거짓말(환각)을 했는지 아닌지 팩트 체크를 할 수 있어 신뢰할 수 있습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [105]:
# RAG 파이프라인 구축을 위한 LangChain 코어 패키지 및 의존성 모듈 설치
!pip install -U -q langchain langchain-openai langchain-community langchain-core langchain-text-splitters chromadb tiktoken pypdf

# API 키 및 환경 변수 설정
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

Text splitter 사용을 위한 준비입니다

In [107]:
# OpenAI의 cl100k_base 인코딩을 활용하여 토큰 길이를 측정하는 함수 정의
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

### Step 1 Document loader

In [110]:
# PyPDFLoader를 사용하여 비정형 데이터(PDF)를 로드하고,
# RecursiveCharacterTextSplitter를 통해 문맥 손실 없이 청킹(Chunking) 수행
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 문서 가져오기
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

### Step 2 Text splitters

In [111]:
# 2. 문서 자르기 : "자를 때 칼로 무 자르듯 딱 자르지 말고, 앞뒤 내용이 이어지게 50 정도는 겹치게(overlap) 잘라!" (문맥이 안 끊기게 하는 꿀팁)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len
)
docs = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [112]:
# OpenAI 임베딩 모델로 벡터 변환 후, ChromaDB에 인덱싱 및 적재
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [114]:
# 1. 바코드 기계 준비
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [116]:
# 2. 크로마(Chroma) 창고에 재료와 바코드 기계 넣기
db = Chroma.from_documents(docs, embedding_model)

### Step 4 Retrievers

In [118]:
# LLM 초기화 및 MMR 기반 Retriever를 결합한 QA Chain 구성
from langchain_openai import ChatOpenAI
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [122]:
# 1. 똑똑한 비서(AI) 부르기
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

In [121]:
# 2. 비서에게 '창고 뒤지는 방법' 알려주기
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_type="mmr", search_kwargs={"k": 3})
)

### Step 5 Question Answering

In [129]:
query = "데미안의 책내용을 정리해줘"
result = qa.invoke(query)

"데미안"은 독일 작가 헤르만 헤세가 쓴 소설로, 주인공 싱클레어의 성장 과정을 다루고 있습니다. 싱클레어는 어린 시절부터 선과 악, 그리고 자기 정체성에 대한 고민을 하며 성장합니다. 그는 학교에서 데미안이라는 신비로운 소년을 만나게 되는데, 데미안은 싱클레어에게 새로운 시각과 철학을 제시하며 그의 내면 세계에 큰 영향을 미칩니다.

소설은 싱클레어가 자신의 내면의 갈등을 해결하고 진정한 자아를 찾기 위해 노력하는 과정을 그립니다. 데미안은 싱클레어에게 기존의 도덕적 가치관을 넘어서 자신의 길을 찾도록 격려하며, 싱클레어는 이를 통해 성숙해집니다. 작품은 인간의 내면 탐구와 자아 발견의 중요성을 강조하며, 상징적이고 철학적인 요소가 많이 포함되어 있습니다.

In [131]:
query = "데미안의 책내용을 5000자 내외로 정리해줘"
result = qa.invoke(query)

"데미안"은 독일 작가 헤르만 헤세의 소설로, 주인공 에밀 싱클레어의 성장 과정을 다루고 있습니다. 이야기는 싱클레어가 어린 시절 친구 프란츠 크로머에게 협박을 당하면서 시작됩니다. 크로머는 싱클레어에게 돈을 요구하며 그를 괴롭히고, 싱클레어는 두려움에 빠집니다. 이때 싱클레어는 학교에서 막스 데미안이라는 신비로운 소년을 만나게 됩니다.

데미안은 싱클레어에게 큰 영향을 미치며, 그에게 새로운 시각과 사고방식을 제시합니다. 특히, 성경의 카인과 아벨 이야기를 통해 기존의 도덕적 가치관을 재고하게 만듭니다. 데미안은 카인의 표식을 긍정적으로 해석하며, 사회의 규범에 얽매이지 않고 자신의 길을 찾으라고 조언합니다.

싱클레어는 데미안과의 교류를 통해 내면의 갈등과 혼란을 극복하고, 자신의 정체성을 찾아가는 과정을 겪습니다. 그는 점차 자신의 내면 세계를 탐구하며, 기존의 가치관을 넘어서는 새로운 깨달음을 얻게 됩니다. 이러한 과정에서 싱클레어는 데미안뿐만 아니라 데미안의 어머니 에바 부인과도 깊은 유대감을 형성하게 됩니다.

소설은 싱클레어가 자신의 내면의 어둠과 빛을 모두 받아들이고, 진정한 자아를 찾아가는 여정을 그립니다. "데미안"은 인간의 내면 탐구와 자아 발견을 주제로 하며, 독자에게 깊은 철학적 사유를 불러일으키는 작품입니다.

In [123]:
# 3. 드디어 질문하기!
query = "데미안은 어떻게 생겼어?"
result = qa.invoke(query)

데미안은 "결단력 있는 입과 넓은 이마의 독특한 광채"를 가진 지적인 얼굴을 가지고 있다고 묘사됩니다. 그는 또한 "탄력 있는 걸음걸이와 곧은 자세"를 가지고 있으며, 그의 목에서는 신선하고 섬세한 비누 향이 난다고도 언급됩니다. 이러한 묘사는 데미안이 매력적이고 카리스마 있는 인물임을 나타냅니다.

In [124]:
query = "데미안이 싱클레어에게 설명한 '밝은 세계'와 '어두운 세계'는 각각 어떤 특징을 가지고 있어?"
result = qa.invoke(query)

데미안에서 싱클레어가 설명한 '밝은 세계'는 순수하고, 아름답고, 질서 정연한 삶을 강조하는 세계입니다. 이 세계에서는 고백, 용서, 사랑, 경외심, 지혜, 성경 읽기 등이 중요한 요소로 여겨지며, 삶이 깨끗하고 흠 없이 유지되어야 한다고 가르칩니다.

반면 '어두운 세계'는 완전히 다른 특성을 가지고 있습니다. 이 세계는 하인, 일꾼, 유령 이야기, 스캔들, 도살장, 감옥, 술 취한 사람들, 싸움, 범죄, 자살 등의 요소로 가득 차 있으며, 소란스럽고 폭력적인 면모를 가지고 있습니다. 이 세계는 매혹적이면서도 무서운 요소들로 가득 차 있으며, 싱클레어의 집 밖에서 쉽게 접할 수 있는 세계입니다. 이 두 세계는 서로 밀접하게 맞닿아 있으며, 싱클레어는 이 두 세계 사이를 오가며 경험합니다.

In [125]:
query = "압락사스(Abraxas)라는 신은 싱클레어의 내면적 성장에 어떤 상징적인 역할을 했어?"
result = qa.invoke(query)

압락사스(Abraxas)는 싱클레어의 내면적 성장에서 중요한 상징적 역할을 합니다. 압락사스는 선과 악, 신성함과 세속성을 모두 아우르는 신으로, 싱클레어가 자신의 정체성을 찾고 자아를 이해하는 과정에서 중요한 역할을 합니다. 싱클레어는 압락사스를 통해 자신의 내면의 복잡성과 모순을 받아들이고, 그것을 통해 자기 자신을 더 깊이 이해하게 됩니다. 이 신은 싱클레어가 단순한 도덕적 기준을 넘어 자신의 본질을 탐구하고, 자신의 길을 찾도록 돕는 상징으로 작용합니다.

In [126]:
query = "데미안의 얼굴 묘사(나이도, 성별도 모호하고 시간을 초월한 듯한 모습)는 어떤 의미를 담고 있는 거야?"
result = qa.invoke(query)

데미안의 얼굴 묘사는 여러 가지 의미를 담고 있습니다. 이 얼굴은 반은 남성적이고 반은 여성적인 특징을 가지고 있으며, 나이도 모호하고 시간을 초월한 듯한 모습입니다. 이러한 묘사는 데미안이 단순한 개인이 아니라 주인공에게 깊은 영향을 미치는 상징적 존재임을 나타냅니다. 그는 주인공에게 내면의 세계와 자기 발견의 여정을 상징하는 인물로, 그의 얼굴은 주인공이 찾고자 하는 진리와 자기 자신을 반영하는 거울과 같은 역할을 합니다. 이 얼굴은 주인공에게 어떤 메시지를 전달하고 요구를 하는 듯한 느낌을 주며, 이는 주인공이 자신의 정체성과 삶의 의미를 탐구하는 과정에서 중요한 역할을 합니다.

In [127]:
query = "카인과 아벨의 이야기에 대해 데미안은 일반적인 사람들과 어떻게 다르게 해석했어?"
result = qa.invoke(query)

데미안은 카인과 아벨의 이야기를 일반적인 해석과 다르게 보았습니다. 일반적으로 카인은 아벨을 죽인 악한 인물로 여겨지지만, 데미안은 카인을 용기 있고 독특한 사람으로 보았습니다. 그는 카인과 그의 후손들이 다른 사람들과 다르다는 이유로 두려움을 불러일으켰고, 그래서 사람들은 그들에게 별명을 붙이고 신화를 만들어냈다고 설명합니다. 데미안은 카인이 강한 사람이었고, 약한 사람을 죽였다는 점에서 이야기가 사실일 수 있지만, 그가 악한 사람이라는 해석은 잘못되었다고 봅니다. 그는 이 이야기가 두려움에서 비롯된 소문에 기반을 두고 있으며, 카인이 특별한 표식을 가졌다는 점에서 진실일 수 있다고 말합니다.

In [128]:
query = "새가 알을 깨고 나오는 과정을 통해 이 책이 전달하고자 하는 가장 핵심적인 메시지는 무엇이야?"
result = qa.invoke(query)

이 책에서 새가 알을 깨고 나오는 과정은 개인의 성장과 자기 발견을 상징합니다. 주인공은 자신의 내면의 목소리를 듣고, 사회적 규범과 기대를 넘어 자신의 진정한 자아를 찾기 위해 노력합니다. 이 과정은 고통스럽고 도전적일 수 있지만, 결국에는 개인의 자유와 자아 실현을 이루는 데 중요한 단계로 묘사됩니다. 따라서 이 책의 핵심 메시지는 자기 발견과 개인의 성장, 그리고 진정한 자아를 찾기 위한 내적 탐구의 중요성입니다.

In [132]:
query = "데미안을 읽고 나의 정체성을 어떻게 찾을 수 있을까?"
result = qa.invoke(query)

"데미안"은 주인공 싱클레어가 자신의 정체성을 찾아가는 과정을 그린 소설입니다. 이 소설을 통해 자신의 정체성을 찾는 데 도움을 받을 수 있는 몇 가지 방법을 생각해볼 수 있습니다.

1. **내면의 목소리에 귀 기울이기**: 싱클레어는 자신의 내면의 목소리를 듣고 그것을 따르려고 노력합니다. 자신의 진정한 욕구와 열망이 무엇인지 깊이 생각해보세요.

2. **자신의 경험을 탐구하기**: 싱클레어는 다양한 경험을 통해 자신을 이해하려고 합니다. 새로운 경험을 통해 자신에 대해 더 많이 배울 수 있습니다.

3. **멘토나 롤모델 찾기**: 싱클레어에게 데미안은 중요한 멘토이자 롤모델입니다. 당신에게 영감을 주고 지침이 될 수 있는 사람을 찾는 것도 도움이 될 수 있습니다.

4. **자신의 가치와 신념 탐색하기**: 싱클레어는 자신의 가치와 신념을 탐구하며 성장합니다. 당신이 중요하게 여기는 것이 무엇인지 생각해보고 그것을 중심으로 삶을 설계해보세요.

5. **자기 수용과 성장**: 자신의 불완전함을 받아들이고 그것을 통해 성장하려는 자세가 중요합니다. 자신을 있는 그대로 받아들이고 발전할 수 있는 방향을 모색하세요.

이러한 방법들을 통해 자신의 정체성을 찾는 여정을 시작할 수 있을 것입니다.

# 데이만의 얼굴을 그림으로 보고 싶을 때.
( 우리가 연결한 시스템인 RetrievalQA 체인은 오직 텍스트 검색과 텍스트 생성에만 최적화된 길(Path)입니다. 그림을 반환하려면 이미지 생성용 API(예: OpenAI의 DALL-E 3)를 호출하는 새로운 길을 하나 더 만들어 주어야 한다.)

외부 웹 서버에 호스팅되어 있는 정적 리소스(이미지 파일)의 URL을 파이썬 환경으로 불러오는 과정. 텍스트 환경인 주피터 노트북(또는 코랩) UI에서 이미지를 렌더링(Rendering)하기 위해 IPython.display 모듈을 사용하여 시각적으로 출력.



In [134]:
# 1. 내가 찾은 멋진 그림의 주소를 예쁜 상자에 담는다
image_url = "https://cdn.discordapp.com/attachments/1298625374577496094/1509044816317775892/DEMIAN.jpg?ex=6a17bf00&is=6a166d80&hm=d1dddd7eeb7d34c2f60bc0f379b0ea664cebd848538069a6f7d8a66308ed22ea&"

# 2. 그림을 화면에 걸기 위해 '액자' 도구를 사옵니다.
from IPython.display import Image, display

# 3. 그림 주소를 액자(Image)에 끼우고, 화면에 보여달라고(display) 명령합니다!
display(Image(url=image_url))

#그림을 그리는 화가(DALL-E)를 우리 시스템에 어떻게 추가할 수 있을까?


굵은 텍스트

In [136]:
# 수정 완료된 전체 코드
painter = DallEAPIWrapper(model="dall-e-3")
image_link = painter.run("새가 알을 깨고 나오는 신비로운 그림을 그려줘")
print(image_link)

BadRequestError: Error code: 400 - {'error': {'message': "The model 'dall-e-3' does not exist.", 'type': 'image_generation_user_error', 'param': 'model', 'code': 'invalid_value'}}